# Calendar Dimension Table

Builds the standard calendar dimension `adb_amnuay.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31. Weeks follow ISO 8601 (Monday start).

In [0]:
%sql
CREATE OR REPLACE TABLE adb_amnuay.silver.calendar
COMMENT 'Standard calendar dimension, one row per day from 1940-01-01 to 2030-12-31. Weeks follow ISO 8601 (Monday = 1).'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)                   AS date_key,
  d                                                         AS date,
  -- Year
  year(d)                                                   AS year,
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END                 AS half_year,
  concat(year(d), '-H', CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END) AS year_half,
  -- Quarter
  quarter(d)                                                AS quarter,
  concat('Q', quarter(d))                                   AS quarter_name,
  concat(year(d), '-Q', quarter(d))                         AS year_quarter,
  -- Month
  month(d)                                                  AS month,
  date_format(d, 'MMMM')                                    AS month_name,
  date_format(d, 'MMM')                                     AS month_short_name,
  date_format(d, 'yyyy-MM')                                 AS year_month,
  year(d) * 100 + month(d)                                  AS year_month_key,
  -- ISO week
  weekofyear(d)                                             AS week_of_year,
  extract(YEAROFWEEK FROM d)                                AS iso_year,
  concat(extract(YEAROFWEEK FROM d), '-W', lpad(CAST(weekofyear(d) AS STRING), 2, '0')) AS iso_year_week,
  CAST(ceil(dayofmonth(d) / 7.0) AS INT)                    AS week_of_month,
  -- Day
  dayofmonth(d)                                             AS day_of_month,
  dayofyear(d)                                              AS day_of_year,
  extract(DOW_ISO FROM d)                                   AS day_of_week,
  date_format(d, 'EEEE')                                    AS day_name,
  date_format(d, 'EEE')                                     AS day_short_name,
  -- Period boundaries
  CAST(date_trunc('WEEK', d) AS DATE)                       AS week_start_date,
  date_add(CAST(date_trunc('WEEK', d) AS DATE), 6)          AS week_end_date,
  trunc(d, 'MM')                                            AS month_start_date,
  last_day(d)                                               AS month_end_date,
  trunc(d, 'QUARTER')                                       AS quarter_start_date,
  last_day(add_months(trunc(d, 'QUARTER'), 2))              AS quarter_end_date,
  trunc(d, 'YEAR')                                          AS year_start_date,
  make_date(year(d), 12, 31)                                AS year_end_date,
  dayofmonth(last_day(d))                                   AS days_in_month,
  -- Flags
  extract(DOW_ISO FROM d) IN (6, 7)                         AS is_weekend,
  extract(DOW_ISO FROM d) NOT IN (6, 7)                     AS is_weekday,
  d = trunc(d, 'MM')                                        AS is_month_start,
  d = last_day(d)                                           AS is_month_end,
  d = trunc(d, 'QUARTER')                                   AS is_quarter_start,
  d = last_day(add_months(trunc(d, 'QUARTER'), 2))          AS is_quarter_end,
  d = trunc(d, 'YEAR')                                      AS is_year_start,
  d = make_date(year(d), 12, 31)                            AS is_year_end,
  (year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0 AS is_leap_year
FROM dates

In [0]:
%sql
SELECT
  count(*)                         AS row_count,
  count(DISTINCT date_key)         AS distinct_keys,
  datediff(max(date), min(date)) + 1 AS expected_rows,
  min(date)                        AS min_date,
  max(date)                        AS max_date,
  sum(CAST(is_leap_year AND month = 2 AND day_of_month = 29 AS INT)) AS leap_days
FROM adb_amnuay.silver.calendar